In [7]:
%env ANYWIDGET_HMR=1
import pandas as pd
from guidepost import Guidepost
# from campsite import Campsite
from IPython.display import HTML
import json

gp = Guidepost(list_columns=["LOCATION"])
# cs = Campsite()

env: ANYWIDGET_HMR=1


In [ ]:
# --- Synthetic seriation check -------------------------------------------------
# Two node clusters (a*, b*) whose nodes co-occur WITHIN a cluster but never
# across, plus a rare node `rare_x` that almost always runs with the common
# node `a0`. Expected on the heatmap with x = nodes:
#   * cluster A and cluster B each render as adjacent blocks (seriation), and
#   * `rare_x` survives as its own column sitting next to `a0` (association-aware).
#
# NOTE: seriation runs in the Python package (guidepost/seriation.py).
# ANYWIDGET_HMR hot-reloads only the JS bundle, NOT Python — so after pulling the
# seriation changes you must RESTART THE KERNEL for category_order/score to
# appear. The guard below says so explicitly instead of raising a KeyError.
import numpy as np
import pandas as pd
from guidepost import Guidepost

rng = np.random.default_rng(0)
cluster_a = [f'a{i}' for i in range(6)]
cluster_b = [f'b{i}' for i in range(6)]

def _job(nodes):
    return {
        'nodes':   ','.join(map(str, nodes)),   # comma-joined, like real LOCATION data
        'runtime': float(rng.gamma(2, 30)),
        'wait':    float(rng.gamma(2, 60)),
        'util':    float(rng.random()),
        'queue':   rng.choice(['short', 'medium', 'long']),
        'project': rng.choice(['proj1', 'proj2']),
    }

rows = []
for _ in range(120):                                    # within-cluster-A jobs
    k = int(rng.integers(2, 5))
    rows.append(_job(rng.choice(cluster_a, size=k, replace=False)))
for _ in range(120):                                    # within-cluster-B jobs
    k = int(rng.integers(2, 5))
    rows.append(_job(rng.choice(cluster_b, size=k, replace=False)))
for _ in range(4):                                      # rare node, always with common a0
    rows.append(_job(['a0', 'rare_x']))

syn_df = pd.DataFrame(rows)

gp_syn = Guidepost(list_columns=['nodes'])
gp_syn.vis_configs = {
    'x': 'nodes', 'y': 'runtime', 'color': 'util', 'color_agg': 'avg',
    'categorical': 'queue', 'facet_by': 'project',
}
gp_syn.records = syn_df

# Peek at the shipped seriation order + scores (rare_x should score high, sit by a0):
_ss = gp_syn._summary_stats['nodes']
if 'category_order' in _ss:
    print('seriated order:', _ss['category_order'])
    print('rare_x score:', round(_ss['category_score']['rare_x'], 3),
          '| a0 score:', round(_ss['category_score']['a0'], 3))
else:
    print('No category_order on _summary_stats["nodes"] — the kernel is running an '
          'older guidepost build.\nRestart the kernel (Kernel > Restart) and re-run; '
          'HMR reloads only the JS bundle, not the Python package.')

gp_syn


In [9]:
# jobs_data = pd.read_parquet("../data/test_data_med.parquet")
# jobs_data = pd.read_parquet("../data/kestrel_data_2024_01_28_subsample.parquet")
# jobs_data = pd.read_csv("../data/ANL-ALCF-DJC-THETA_20240101_20241231.csv.gz", compression='gzip')
jobs_data = pd.read_csv("../data/ANL-ALCF-DJC-POLARIS_20250101_20251231.csv.gz", compression='gzip')
# jobs_data = pd.read_csv("../data/campsite-data/ANL-ALCF-DJC-AURORA_20260101_20260216.csv.gz", compression='gzip')
# df = pd.read_csv("../data/ANL-ALCF-MACHINESTATUS-POLARIS_20250101_20251231.csv.gz", compression='gzip')

# gpu_data = pd.read_csv("../data/campsite-data/ANL-ALCF-GPU-NODE-POLARIS_20260101_20260311_smol.csv.gz", compression='gzip')

# jobs_data
gp.records = jobs_data
# cs.records = jobs_data

/var/folders/d8/rxc9f9bn0mn4tv6vr6hwk1ww0000gn/T/ipykernel_4245/1522435336.py:4: DtypeWarning: Columns (0: MODE) have mixed types. Specify dtype option on import or set low_memory=False.
  jobs_data = pd.read_csv("../data/ANL-ALCF-DJC-POLARIS_20250101_20251231.csv.gz", compression='gzip')


In [6]:
gp

In [4]:
%env ANYWIDGET_HMR=1
import pandas as pd
from campsite import Campsite

cs = Campsite()

jobs_data = pd.read_csv("../data/ANL-ALCF-DJC-POLARIS_20250101_20251231.csv.gz", compression='gzip')

cs.records = jobs_data

env: ANYWIDGET_HMR=1


ModuleNotFoundError: No module named 'campsite'

In [3]:
hyp = pd.read_csv("../data/campsite-data/Hypothesis Generation/Generated Hypotheses/hyp.uncertain.kestrel.csv")
# cs.extract_nl_invariants(hyp, '../data/campsite-data/ANL-AURORA-new/')

In [4]:
hyp

,persona,hypothesis_id,hypothesis_text
0,System Administrator,SA1,The 90th percentile of queue_wait for qos = hi...
1,System Administrator,SA2,The estimated reduction in median queue_wait f...
2,System Administrator,SA3,The variance of queue_wait for qos = standby i...
3,System Administrator,SA4,The probability that the top 5% of queue_wait ...
4,System Administrator,SA5,The estimated probability that a standard part...
5,System Administrator,SA6,At least one partition has an estimated 95th p...
6,System Administrator,SA7,The estimated share of submitted jobs accounte...
7,System Administrator,SA8,The coefficient of variation of user-level med...
8,System Administrator,SA9,There is high probability that users in the to...
9,System Administrator,SA10,The variance of queue_wait in the shared parti...


In [3]:
cs

Campsite()

## Translated Hypothesis
Let Y_i = QUEUED_WAIT_SECONDS_i / RUNTIME_SECONDS_i for job i.
Let D10 be the 10th percentile of RUNTIME_SECONDS and M be the median of RUNTIME_SECONDS.
Define A = {i: RUNTIME_SECONDS_i ≤ D10} and B = {i: RUNTIME_SECONDS_i > M}.

H0: median(Y_i : i ∈ A) ≤ median(Y_i : i ∈ B)
H1: median(Y_i : i ∈ A) > median(Y_i : i ∈ B)

In [9]:
def build_vega_fusion_chart():
    import altair as alt
    # Enable Vega fusion data transformation as requested
    alt.data_transformers.enable("vegafusion")

    # Base dataset reference (named dataset as in the Vega-Lite spec)
    base = alt.Chart(alt.Data(name="DATASET"))

    # Layer 1: scatter points of Y by Group (A/B proxy)
    points = base.transform_calculate(
        Y="datum.QUEUED_WAIT_SECONDS / datum.RUNTIME_SECONDS",
        Group="datum.RUNTIME_SECONDS <= 10 ? 'A' : (datum.RUNTIME_SECONDS > 274 ? 'B' : 'Other')"
    ).transform_filter(
        "datum.RUNTIME_SECONDS > 0 && (datum.Group === 'A' || datum.Group === 'B')"
    ).mark_point(opacity=0.4).encode(
        x=alt.X("Group:N", axis=alt.Axis(title="Group (proxy for A vs B)")),
        y=alt.Y("Y:Q"),
        color="Group:N"
    )

    # Layer 2: median of Y by Group
    bars = base.transform_calculate(
        Y="datum.QUEUED_WAIT_SECONDS / datum.RUNTIME_SECONDS",
        Group="datum.RUNTIME_SECONDS <= 10 ? 'A' : (datum.RUNTIME_SECONDS > 274 ? 'B' : 'Other')"
    ).transform_filter(
        "datum.RUNTIME_SECONDS > 0 && (datum.Group === 'A' || datum.Group === 'B')"
    ).transform_aggregate(
        median_Y="median(Y)",
        groupby=["Group"]
    ).mark_bar().encode(
        x="Group:N",
        y="median_Y:Q",
        color="Group:N"
    )

    # Layer the two charts
    chart = alt.layer(points, bars)
    return chart

## Translated Hypothesis
H0: med{QUEUED_WAIT_SECONDS / RUNTIME_SECONDS : RUNTIME_SECONDS ≤ Q_{0.10}(RUNTIME_SECONDS)} ≤ med{QUEUED_WAIT_SECONDS / RUNTIME_SECONDS : RUNTIME_SECONDS > Q_{0.50}(RUNTIME_SECONDS)}

Ha: med{QUEUED_WAIT_SECONDS / RUNTIME_SECONDS : RUNTIME_SECONDS ≤ Q_{0.10}(RUNTIME_SECONDS)} > med{QUEUED_WAIT_SECONDS / RUNTIME_SECONDS : RUNTIME_SECONDS > Q_{0.50}(RUNTIME_SECONDS)}

In [8]:
def vega_fusion_altair_plot(data):
    import altair as alt
    alt.data_transformers.enable("vegafusion")

    # Pre-compute fields that transform_calculate would create
    df = data.copy()
    df['Group'] = df['RUNTIME_SECONDS'].apply(
        lambda x: 'Low_RTN' if x <= 21 else ('High_RTN' if x > 274 else None)
    )
    df['ratio'] = df.apply(
        lambda r: None if r['RUNTIME_SECONDS'] == 0 else r['QUEUED_WAIT_SECONDS'] / r['RUNTIME_SECONDS'],
        axis=1
    )
    df = df.dropna(subset=['Group', 'ratio'])

    layer1 = alt.Chart(df).mark_point().encode(
        x=alt.X('Group:N', axis=alt.Axis(title='Runtime Group')),
        y=alt.Y('ratio:Q', aggregate='median', axis=alt.Axis(title='median_ratio')),
        color=alt.Color('Group:N', legend=alt.Legend(title='Group'))
    )

    layer2 = alt.Chart(df).mark_bar(opacity=0.6).encode(
        x=alt.X('ratio:Q', bin=alt.Bin(maxbins=20)),
        y=alt.Y('count():Q'),
        color=alt.Color('Group:N', legend=alt.Legend(title='Group'))
    )

    return layer1 

vega_fusion_altair_plot(jobs_data)

alt.Chart(...)

In [4]:
# gp.vis_configs = {
#         'x': 'QUEUED_TIMESTAMP',
#         'y': 'RUNTIME_SECONDS',
#         'color': 'NODES_REQUESTED',
#         'color_agg': 'avg',
#         'categorical': 'USERNAME_GENID',
#         'facet_by': 'QUEUE_NAME'
# }

In [ ]:
gp

In [25]:
gp.selection

,JOB_NAME,COBALT_JOBID,MACHINE_NAME,QUEUED_TIMESTAMP,QUEUED_DATE_ID,START_TIMESTAMP,START_DATE_ID,END_TIMESTAMP,END_DATE_ID,USERNAME_GENID,...,OVERBURN_CORE_HOURS,IS_OVERBURN,GPUS_REQUESTED,IS_PYTHON,PYTHON_EXECUTABLE_PATH,PYTHON_EXECUTABLE_VERSION,TASK_EXIT_CODES,IS_TASK_EXIT_CODES_NON_ZERO,SCIENCE_FIELD,SCIENCE_FIELD_SHORT
151390,5650314.polaris,0,polaris,2025-07-25 15:05:04,20250725,2025-07-25 15:05:13,20250725,2025-07-25 15:15:43,20250725,70555974529383,...,0.0,0,8,0,NaN,NaN,NaN,-1,Chemistry:Catalytic,Chemistry
151443,5656977.polaris,0,polaris,2025-07-25 16:09:04,20250725,2025-07-25 16:11:22,20250725,2025-07-25 16:21:45,20250725,70555974529383,...,0.0,0,4,0,NaN,NaN,NaN,-1,Chemistry:Catalytic,Chemistry
151482,5658074.polaris,0,polaris,2025-07-25 16:41:09,20250725,2025-07-25 16:41:22,20250725,2025-07-25 16:53:14,20250725,70555974529383,...,0.0,0,8,0,NaN,NaN,NaN,-1,Chemistry:Catalytic,Chemistry
151692,5678695.polaris,0,polaris,2025-07-25 21:01:46,20250725,2025-07-25 21:05:33,20250725,2025-07-25 21:05:48,20250725,70555974529383,...,0.0,0,4,0,NaN,NaN,NaN,-1,Chemistry:Catalytic,Chemistry
151787,5679066.polaris,0,polaris,2025-07-25 21:05:34,20250725,2025-07-25 22:26:49,20250725,2025-07-25 22:57:37,20250725,70555974529383,...,0.0,0,44,0,NaN,NaN,NaN,-1,Chemistry:Catalytic,Chemistry
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173520,5898761.polaris,0,polaris,2025-08-16 11:49:59,20250816,2025-08-16 12:11:40,20250816,2025-08-16 12:12:17,20250816,70555974529383,...,0.0,0,4,0,NaN,NaN,NaN,-1,Chemistry:Catalytic,Chemistry
173523,5898765.polaris,0,polaris,2025-08-16 12:12:07,20250816,2025-08-16 12:12:14,20250816,2025-08-16 12:29:50,20250816,70555974529383,...,0.0,0,44,0,NaN,NaN,NaN,-1,Chemistry:Catalytic,Chemistry
173524,5898763.polaris,0,polaris,2025-08-16 12:12:06,20250816,2025-08-16 12:12:12,20250816,2025-08-16 12:29:53,20250816,70555974529383,...,0.0,0,44,0,NaN,NaN,NaN,-1,Chemistry:Catalytic,Chemistry
173525,5898764.polaris,0,polaris,2025-08-16 12:12:07,20250816,2025-08-16 12:12:14,20250816,2025-08-16 12:29:54,20250816,70555974529383,...,0.0,0,44,0,NaN,NaN,NaN,-1,Chemistry:Catalytic,Chemistry


In [13]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

base = Path("../data/campsite-data/esif.hpc.kestrel.job-anon/esif.hpc.kestrel.job-anon/year=2025/")
files = sorted(base.rglob("*.parquet"))

def normalize_types(table):
    """Cast all timestamp columns to UTC and all duration columns to microseconds."""
    for i, field in enumerate(table.schema):
        if pa.types.is_timestamp(field.type) and field.type.tz and field.type.tz != "UTC":
            col = table.column(i).cast(pa.timestamp(field.type.unit, tz="UTC"))
            table = table.set_column(i, field.with_type(col.type), col)
        elif pa.types.is_duration(field.type) and field.type.unit != "us":
            col = table.column(i).cast(pa.duration("us"))
            table = table.set_column(i, field.with_type(col.type), col)
    return table

tables = [normalize_types(pq.read_table(f)) for f in files]
table = pa.concat_tables(tables, promote_options="default")
df = table.to_pandas(types_mapper=pd.ArrowDtype)
df

ArrowInvalid: Must pass at least one table

In [ ]:
jobs_data['PROJECT_NAME_GENID'].unique()

In [28]:
cs.records = jobs_data

Note: The following columns contain missing values (count per column): [MODE(158854)].


In [3]:
cs

Campsite()

In [30]:
result = cs.check_pipeline("Within comparable job sizes (grouped by quartiles of NODES_REQUESTED), median QUEUED_WAIT_SECONDS differs across QUEUE_NAME.")
print(result["ir"])            # parsed IR AST
print(result["code"])          # generated Python code
print(result["vega_lite_spec"]) # vega-lite spec
for failure in result["failures"]:
    print(failure)


{'type': 'hypothesis', 'event': {'type': 'comparison', 'quantity': {'type': 'error', 'boundary': 'quantity', 'message': "Parse error in quantity: No terminal matches '[' in the current parser context, at line 1 col 3", 'text': "(E[QUEUED_WAIT_SECONDS (QUEUE_NAME = 'debug')] - E[QUEUED_WAIT_SECONDS (QUEUE_NAME = 'preemptable')])", 'start': 0, 'end': 102}, 'comparator': '!=', 'referent': {'type': 'const', 'value': 0.0}}, 'across_partition': None, 'within_partition': None}
def plot_mean_queued_wait_spec():
    import altair as alt
    alt.data_transformers.enable("vegafusion")

    data = alt.Data(name="data")

    chart = (
        alt.Chart(data)
        .mark_point()
        # Filter to only the two queues
        .transform_filter("datum.QUEUE_NAME === 'debug' || datum.QUEUE_NAME === 'preemptable'")
        # Bin NODES_REQUESTED into NODES_REQUESTED_BIN with up to 20 bins
        .transform_bin('NODES_REQUESTED', as_='NODES_REQUESTED_BIN', maxbins=20)
        # Aggregate: mean QUEUED_

In [38]:
import pandas as pd
import altair as alt

# Load data
df = jobs_data

# Keep only needed columns
df = df.dropna(subset=["NODES_REQUESTED", "QUEUED_WAIT_SECONDS", "QUEUE_NAME"]).copy()

# Build quantile bins, allowing duplicate edges to collapse
q = pd.qcut(df["NODES_REQUESTED"], q=4, duplicates="drop")

# Turn interval categories into plain ordered labels that Altair can sort safely
intervals = list(q.cat.categories)
label_map = {interval: f"Q{i+1}: {interval.left:g}–{interval.right:g} nodes"
             for i, interval in enumerate(intervals)}

df["NODES_SIZE_GROUP"] = q.map(label_map).astype(str)

# Explicit facet order
facet_order = [label_map[i] for i in intervals]

# Optional: reduce extreme skew in wait time display
# Uncomment this if raw seconds are too stretched
# df["WAIT_DISPLAY"] = df["QUEUED_WAIT_SECONDS"].clip(
#     upper=df["QUEUED_WAIT_SECONDS"].quantile(0.99)
# )
# y_field = "WAIT_DISPLAY:Q"
# y_title = "Queued Wait Time (seconds, capped at 99th pct)"

y_field = "QUEUED_WAIT_SECONDS:Q"
y_title = "Queued Wait Time (seconds)"

chart = (
    alt.Chart(df)
    .mark_boxplot(extent="min-max")
    .encode(
        x=alt.X(
            "QUEUE_NAME:N",
            title="Queue Name",
            sort="ascending"
        ),
        y=alt.Y(
            y_field,
            title=y_title
        ),
        color=alt.Color("QUEUE_NAME:N", legend=None)
    )
    .properties(width=180, height=300)
    .facet(
        column=alt.Column(
            "NODES_SIZE_GROUP:N",
            title="Comparable Job Sizes",
            sort=facet_order
        )
    )
    .resolve_scale(y="shared")
    .properties(
        title="Queued Wait Time by Queue, Within Comparable Job-Size Groups"
    )
)

chart


# def plot_mean_queued_wait_spec():
#     import altair as alt
#     alt.data_transformers.enable("vegafusion")

#     data = jobs_data

#     chart = (
#         alt.Chart(data)
#         .mark_point()
#         # Filter to only the two queues
#         .transform_filter("datum.QUEUE_NAME === 'debug' || datum.QUEUE_NAME === 'preemptable'")
#         # Bin NODES_REQUESTED into NODES_REQUESTED_BIN with up to 20 bins
#         .transform_bin('NODES_REQUESTED', as_='NODES_REQUESTED_BIN', maxbins=20)
#         # Aggregate: mean QUEUED_WAIT_SECONDS by NODES_REQUESTED_BIN and QUEUE_NAME
#         .transform_aggregate(
#             mean_QUEUED_WAIT_SECONDS='mean(QUEUED_WAIT_SECONDS)',
#             groupby=['NODES_REQUESTED_BIN', 'QUEUE_NAME']
#         )
#         .encode(
#             x=alt.X('NODES_REQUESTED_BIN:O', axis=alt.Axis(title='NODES_REQUESTED (binned)')),
#             y=alt.Y('mean_QUEUED_WAIT_SECONDS:Q', axis=alt.Axis(title='Mean QUEUED_WAIT_SECONDS')),
#             color=alt.Color('QUEUE_NAME:N', legend=alt.Legend(title='QUEUE_NAME')),
#             tooltip=None
#         )
#         .properties(width=600, height=300)
#     )

#     return chart

# plot_mean_queued_wait_spec()

alt.FacetChart(...)

In [ ]:
print(result["ir"], "\n")            # parsed IR AST
print(result["code"], "\n")          # generated Python code
print(result["vega_lite_spec"], "\n") # vega-lite spec

for failure in result["failures"]:
    print(failure)

'hypothesis   :- E[QUEUED_WAIT_SECONDS | QUEUE_NAME = ANYLEVEL ^ NODES_REQUESTED BETWEEN (1, 1)] - E[QUEUED_WAIT_SECONDS | QUEUE_NAME = "debug" ^ NODES_REQUESTED BETWEEN (1, 1)] != 0'


In [39]:
def vega_fusion_mean_queued_wait_chart(dataset):
    """
    Returns an Altair chart that matches the provided Vega-Lite spec:
    - Filter NODES_REQUESTED == 1
    - Aggregate mean of QUEUED_WAIT_SECONDS by QUEUE_NAME
    - Filter to QUEUE_NAME in {'tiny', 'debug'}
    - Mark as point
    - X: QUEUE_NAME (nominal) with sort order ['tiny', 'debug']
    - Y: mean_queued_wait (quantitative) with axis title
    - Color by QUEUE_NAME
    - No tooltips
    Note: dataset should be a pandas DataFrame with columns:
    QUEUE_NAME, NODES_REQUESTED, QUEUED_WAIT_SECONDS
    """
    import altair as alt
    alt.data_transformers.enable("vegafusion")

    chart = alt.Chart(dataset).transform_filter(
        alt.datum.NODES_REQUESTED == 1
    ).transform_aggregate(
        mean_queued_wait='mean(QUEUED_WAIT_SECONDS)',
        groupby=['QUEUE_NAME']
    ).transform_filter(
        (alt.datum.QUEUE_NAME == 'tiny') | (alt.datum.QUEUE_NAME == 'debug')
    ).mark_point().encode(
        x=alt.X('QUEUE_NAME:N', sort=['tiny', 'debug']),
        y=alt.Y('mean_queued_wait:Q', axis=alt.Axis(title='Mean QUEUED_WAIT_SECONDS (NODES_REQUESTED=1)')),
        color=alt.Color('QUEUE_NAME:N')
        # Tooltip intentionally omitted to match spec
    )

    # Attempt to display in environments like Jupyter
    try:
        from IPython.display import display
        display(chart)
    except Exception:
        pass

vega_fusion_mean_queued_wait_chart(jobs_data)

alt.Chart(...)

In [ ]:
def viz_queued_wait_seconds_by_queue():
    """
    Create an Altair chart matching the provided Vega-Lite spec:
    - Strip plot of QUEUED_WAIT_SECONDS by QUEUE_NAME (point marks)
    - Overlay mean QUEUED_WAIT_SECONDS per QUEUE_NAME (larger red points)
    - No tooltips
    - Uses vegafusion data transformer
    """
    import altair as alt
    alt.data_transformers.enable("vegafusion")

    # Base chart uses a named dataset (to align with the Vega-Lite spec)
    base = alt.Chart(jobs_data)

    # Layer 1: strip plot (QUEUED_WAIT_SECONDS by QUEUE_NAME)
    layer1 = base.mark_point(size=30, filled=True).encode(
        x=alt.X("QUEUE_NAME:N"),
        y=alt.Y("QUEUED_WAIT_SECONDS:Q"),
        color=alt.Color("QUEUE_NAME:N")
    )

    # Layer 2: mean per QUEUE_NAME (overlay)
    layer2 = base.transform_aggregate(
        aggregate=[{"op": "mean", "field": "QUEUED_WAIT_SECONDS", "as": "MEAN_QUEUED_WAIT_SECONDS"}],
        groupby=["QUEUE_NAME"]
    ).mark_point(size=100, filled=True).encode(
        x=alt.X("QUEUE_NAME:N"),
        y=alt.Y("MEAN_QUEUED_WAIT_SECONDS:Q"),
        color=alt.value("red")
    )

    chart = (layer1 + layer2).resolve_scale(color="independent").properties(
        title="QUEUED_WAIT_SECONDS by QUEUE_NAME with Means (strip plot + mean overlay)"
    )
    return chart

viz_queued_wait_seconds_by_queue()

In [ ]:
# cs.test_server()
# hypothesis = 'E[failure_indicator | utilization < 0.3 ^ gpu = true] > E[failure_indicator | utilization > 0.8 ^ gpu = true]'
natural_language = 'Among queues designed for similar job sizes, one queue exhibits significantly longer wait times than the others.'
# natural_language = 'Jobs requesting GPUs but exhibiting low GPU utilization are more likely to fail than high-utilization GPU jobs.'
# natural_language = 'Resource consumption across projects follows a heavy-tailed distribution.'
# c.load_data(df)
# result = c.test_artifact_gen("The average runtime of MPI jobs exceeds that of OpenMP jobs")
# HTML((cs.test_parser(hypothesis, natural_language)))
response = cs.test_artifact_gen(natural_language)
# print(json.dumps(response['parsed'], indent=2))
# print(json.dumps(response['violations'], indent=2))

In [13]:
df = jobs_data.groupby('QUEUE_NAME').filter(lambda g: len(g) > 1)

In [ ]:
cs

In [ ]:
jobs_data.columns

In [14]:
gp.records = df

In [11]:
gp

Guidepost()

In [ ]:
I am interested in relationships between memory efficiency, power, and job type.

In [ ]:
gp.selection